<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 所著《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第六章：练习题解答

本笔记本中使用的包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## 练习 6.1：添加格式感知的奖励塑形

- 如果未找到"\boxed{}"答案，我们可以使用第3章中编码的`fallback="number_then_full"`回退机制，分配部分奖励（得分0.5）如下：

In [ ]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def partial_reward_rlvr(answer_text, ground_truth):
    
    # 1) Try to extract a boxed answer
    boxed = extract_final_candidate(
        answer_text, fallback=None
    )
    if boxed:
        correct = grade_answer(boxed, ground_truth)
        return 1.0 if correct else 0.0

    # 2) If no boxed answer is found, look for number
    unboxed = extract_final_candidate(
        answer_text, fallback="number_then_full"
    )
    if unboxed:
        correct = grade_answer(unboxed, ground_truth)
        return 0.5 if correct else 0.0

    return 0.0

- 当接入第6章代码并在相同设置下训练时，部分奖励变体方案的准确率（37.8%）低于标准GRPO设置（47.4%），尽管两者平均使用的token数量相近。

| # | 方法                                    | 步长 | 最大token数 | rollout次数 | 准确率 | 平均token数 |
|---|------------------------------------------|------|------------|--------------|----------|----------------|
| 1 | GRPO（第6章）                            | 50   | 512        | 8            | 47.4%    | 586.11         |
| 2 | GRPO部分奖励（练习6.1）                  | 50   | 512        | 8            | 37.8%    | 550.33         |

&nbsp;
## 练习 6.2：零优势情况

- 如果所有奖励都相等（例如，它们全为0或全为1），那么所有优势值都将为0，因为减去均值会消除共享的奖励值，只留下零值，我们可以在下方演示这一点

In [3]:
import torch

rollout_rewards = [0., 0., 0., 0.]
rewards = torch.tensor(rollout_rewards)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([0., 0., 0., 0.])


In [4]:
rollout_rewards = [1., 1., 1., 1.]
rewards = torch.tensor(rollout_rewards)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([0., 0., 0., 0.])


- 现在，如果所有优势值都为零，损失也将为零，因为损失是通过将优势值与对数概率相乘来计算的，而乘以零会消除它们的贡献。

```python
pg_loss = -(advantages.detach() * logps).mean()
```

因此，策略梯度为零，且模型参数不会针对该提示进行更新。

- 这种行为是刻意设计的；如果所有rollouts都同样糟糕或同样优秀，就没有相对信号来告诉模型应该强化或抑制哪种行为
- 直观来说，如果模型正确回答了所有问题，就没有必要更新它
- 反之，如果模型回答所有问题都错误，我们也不希望更新模型来强化这种行为